# Task 3: Real-Time Echocardiogram Video Analysis
Student: Sabeeh | Roll number: 23K-0002

## 1. Imports and configuration
Place one short sample at `data/sample_echo.mp4`. This notebook uses a local MP4 file, not a webcam. A desktop GUI is needed for OpenCV display windows.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

TASK_DIR = Path.cwd()
if TASK_DIR.name != "Task_3_Echo_Analysis" and (TASK_DIR / "Task_3_Echo_Analysis").exists(): TASK_DIR /= "Task_3_Echo_Analysis"
DATA_DIR, OUTPUT_DIR = TASK_DIR / "data", TASK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_PATH = DATA_DIR / "sample_echo.mp4"

## 2. Open sample video and process frames
Per frame: grayscale histogram equalization, JET heatmap, channel-mean balance shift, logarithmic compression, and gamma transform. Gamma below one raises dark values while compressing bright values relative to them.

In [ ]:
if not VIDEO_PATH.is_file():
    raise FileNotFoundError("Place one short EchoNet-Dynamic MP4 sample at Task_3_Echo_Analysis/data/sample_echo.mp4. Do not add the full dataset.")
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    cap.release()
    raise RuntimeError(f"OpenCV could not open {VIDEO_PATH}. Check that it is a valid MP4 with a supported codec.")

GAMMA = 1.4
MAX_FRAMES_TO_SAVE = 1
frame_index = 0
saved_representative = False
gui_available = True
try:

    while True:
        success, frame = cap.read()
        if not success:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        equalized = cv2.equalizeHist(gray)
        heatmap = cv2.applyColorMap(equalized, cv2.COLORMAP_JET)


        heat_f = heatmap.astype(np.float32)
        channel_means = heat_f.mean(axis=(0, 1), keepdims=True)
        neutral_mean = channel_means.mean()
        balanced = np.clip(heat_f + (neutral_mean - channel_means), 0, 255).astype(np.uint8)


        normalized = balanced.astype(np.float32) / 255.0
        logged = np.log1p(normalized) / np.log(2.0)
        enhanced = np.round(np.power(np.clip(logged, 0, 1), GAMMA) * 255).clip(0,255).astype(np.uint8)
        enhanced = cv2.resize(enhanced, (frame.shape[1], frame.shape[0]))
        side_by_side = np.hstack((frame, enhanced))

        if not saved_representative:
            cv2.imwrite(str(OUTPUT_DIR / "representative_raw_frame.png"), frame)
            cv2.imwrite(str(OUTPUT_DIR / "representative_enhanced_frame.png"), enhanced)
            cv2.imwrite(str(OUTPUT_DIR / "side_by_side.png"), side_by_side)
            saved_representative = True
        frame_index += 1
        if gui_available:
            try:
                cv2.imshow("Original | Enhanced echocardiogram (press q to quit)", side_by_side)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
            except cv2.error as error:
                gui_available = False
                print(f"OpenCV display is unavailable; continuing file processing without a window: {error}")
finally:
    cap.release()
    cv2.destroyAllWindows()

if frame_index == 0:
    raise RuntimeError("No frames were decoded from the sample video.")
print(f"Processed {frame_index} frame(s); representative screenshots saved in {OUTPUT_DIR}.")

## 3. Review saved representative output

In [ ]:
raw_saved = cv2.imread(str(OUTPUT_DIR / "representative_raw_frame.png"))
enh_saved = cv2.imread(str(OUTPUT_DIR / "representative_enhanced_frame.png"))
if raw_saved is not None and enh_saved is not None:
    fig, axes = plt.subplots(1,2,figsize=(12,5))
    axes[0].imshow(cv2.cvtColor(raw_saved,cv2.COLOR_BGR2RGB)); axes[0].set_title("Representative raw frame"); axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(enh_saved,cv2.COLOR_BGR2RGB)); axes[1].set_title("Representative enhanced frame"); axes[1].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Run the video processing cell first to generate representative frames.")